# 📰 Notebook 4: News Sentiment Analysis
**FinTech Stock Market Analysis Project**

This notebook covers:
- Scraping financial news headlines via RSS feeds (no API key required)
- Sentiment scoring with VADER (financial-tuned)
- Aggregating daily sentiment scores
- Correlating sentiment with next-day price movement
- Visualizing sentiment trends vs stock prices

In [1]:
import pandas as pd
import numpy as np
import feedparser
import requests
from datetime import datetime, timedelta
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import time
import warnings
warnings.filterwarnings('ignore')

analyzer = SentimentIntensityAnalyzer()

TICKERS = ['JPM', 'V', 'MA', 'PYPL', 'SQ', 'COIN', 'HOOD', 'AFRM']
NAMES   = {'JPM':'JPMorgan','V':'Visa','MA':'Mastercard','PYPL':'PayPal',
            'SQ':'Block','COIN':'Coinbase','HOOD':'Robinhood','AFRM':'Affirm'}

print('✅ Libraries loaded!')

✅ Libraries loaded!


## 1. Collect News Headlines via RSS Feeds

In [2]:
def get_rss_headlines(ticker, company_name, max_articles=50):
    """Fetch headlines from multiple RSS sources for a given stock."""
    feeds = [
        f'https://feeds.finance.yahoo.com/rss/2.0/headline?s={ticker}&region=US&lang=en-US',
        f'https://news.google.com/rss/search?q={company_name}+stock&hl=en-US&gl=US&ceid=US:en',
        f'https://news.google.com/rss/search?q={ticker}+fintech&hl=en-US&gl=US&ceid=US:en',
    ]

    articles = []
    for url in feeds:
        try:
            feed = feedparser.parse(url)
            for entry in feed.entries[:max_articles // len(feeds)]:
                pub = entry.get('published_parsed') or entry.get('updated_parsed')
                date = datetime(*pub[:6]) if pub else datetime.now()
                articles.append({
                    'ticker': ticker,
                    'date': date.date(),
                    'title': entry.get('title', ''),
                    'summary': entry.get('summary', '')
                })
        except Exception as e:
            pass
        time.sleep(0.3)

    return articles

print('Collecting headlines... (this may take ~30 seconds)\n')

all_articles = []
for ticker in TICKERS:
    articles = get_rss_headlines(ticker, NAMES[ticker])
    all_articles.extend(articles)
    print(f'  {ticker}: {len(articles)} articles collected')

news_df = pd.DataFrame(all_articles)
print(f'\n✅ Total articles: {len(news_df)}')
news_df.head(5)


  JPM: 48 articles collected
  V: 48 articles collected
  MA: 48 articles collected
  PYPL: 48 articles collected
  SQ: 32 articles collected
  COIN: 48 articles collected
  HOOD: 48 articles collected
  AFRM: 48 articles collected

✅ Total articles: 368


,ticker,date,title,summary
0,JPM,2026-04-28,Is It Time To Reassess JPMorgan Chase (JPM) Af...,"Before getting into the numbers, it helps to a..."
1,JPM,2026-04-28,Robinhood To Report As Growth Slows And Expans...,Robinhood prepares to report earnings for the ...
2,JPM,2026-04-28,JPMorganChase named first global banking partn...,The deal covers the Los Angeles 2028 and Frenc...
3,JPM,2026-04-28,Campbell Global Closes Acquisition of Sandpipe...,"Campbell Global, a J.P. Morgan company and a l..."
4,JPM,2026-04-28,JPM Off to a Solid 2026 Start: Should Investor...,JPMorgan's Q1 earnings beat expectations on re...


## 2. VADER Sentiment Scoring

In [3]:
def score_sentiment(row):
    """Combine title + summary for richer sentiment signal."""
    text   = f"{row['title']}. {row['summary']}"
    scores = analyzer.polarity_scores(text)
    return scores['compound']  # -1 (very negative) to +1 (very positive)

news_df['sentiment'] = news_df.apply(score_sentiment, axis=1)
news_df['sentiment_label'] = pd.cut(
    news_df['sentiment'],
    bins=[-1, -0.05, 0.05, 1],
    labels=['Negative', 'Neutral', 'Positive']
)

print('📊 Sentiment Distribution:')
print(news_df['sentiment_label'].value_counts())
print(f'\nMean sentiment: {news_df["sentiment"].mean():.4f}')

# Sample headlines
print('\n🔴 Most Negative Headlines:')
print(news_df.nsmallest(3, 'sentiment')[['ticker','title','sentiment']].to_string())
print('\n🟢 Most Positive Headlines:')
print(news_df.nlargest(3, 'sentiment')[['ticker','title','sentiment']].to_string())

📊 Sentiment Distribution:
sentiment_label
Positive    187
Neutral     119
Negative     62
Name: count, dtype: int64

Mean sentiment: 0.2211

🔴 Most Negative Headlines:
    ticker                                                                             title  sentiment
62       V  Digital Payment Update - AI's Double Role: Fueling and Fighting E-Commerce Fraud    -0.9705
155   PYPL  Digital Payment Update - AI's Double Role: Fueling and Fighting E-Commerce Fraud    -0.9705
9      JPM                   JPMorgan Says Firms Avoid Raising Forecasts Due to War Concerns    -0.9683

🟢 Most Positive Headlines:
    ticker                                                                                                                                                                                                title  sentiment
59       V                                                                                                                                          Amazon Is a Strong Bu

## 3. Sentiment by Ticker

In [4]:
sentiment_by_ticker = (
    news_df.groupby('ticker')['sentiment']
    .agg(['mean', 'std', 'count'])
    .round(4)
    .rename(columns={'mean':'Avg Sentiment','std':'Std Dev','count':'Articles'})
    .sort_values('Avg Sentiment', ascending=False)
)

fig = go.Figure(go.Bar(
    x=sentiment_by_ticker.index,
    y=sentiment_by_ticker['Avg Sentiment'],
    marker_color=['green' if v > 0 else 'red' for v in sentiment_by_ticker['Avg Sentiment']],
    text=sentiment_by_ticker['Avg Sentiment'].round(3),
    textposition='outside'
))
fig.add_hline(y=0, line_dash='dash', line_color='white', opacity=0.5)
fig.update_layout(template='plotly_dark', height=400,
                   title='📰 Average News Sentiment by FinTech Stock',
                   yaxis_title='VADER Compound Score')
fig.show()
print(sentiment_by_ticker)

        Avg Sentiment  Std Dev  Articles
ticker                                  
MA             0.3384   0.4461        48
AFRM           0.2849   0.4074        48
COIN           0.2354   0.4385        48
HOOD           0.2346   0.4752        48
JPM            0.2335   0.5135        48
V              0.2250   0.4862        48
PYPL           0.1596   0.4091        48
SQ            -0.0239   0.4931        32


## 4. Sentiment vs Price Movement Correlation

In [5]:
TARGET_TICKER = 'JPM'

# Daily aggregated sentiment
ticker_news = news_df[news_df['ticker'] == TARGET_TICKER].copy()
ticker_news['date'] = pd.to_datetime(ticker_news['date'])

daily_sentiment = (
    ticker_news.groupby('date')['sentiment']
    .agg(['mean', 'count'])
    .rename(columns={'mean':'sentiment_avg', 'count':'article_count'})
)

# Load stock price data
stock_df = pd.read_csv(f'../data/{TARGET_TICKER}_processed.csv',
                        index_col=0, parse_dates=True)
stock_df.index = pd.to_datetime(stock_df.index)

# Merge on date
merged = stock_df[['Close', 'Daily_Return']].join(daily_sentiment, how='inner')
merged['Next_Return'] = merged['Daily_Return'].shift(-1)
merged.dropna(inplace=True)

if len(merged) > 5:
    corr = merged['sentiment_avg'].corr(merged['Next_Return'])
    print(f'\n📊 Sentiment vs Next-Day Return Correlation ({TARGET_TICKER}): {corr:.4f}')

    fig = go.Figure(go.Scatter(
        x=merged['sentiment_avg'], y=merged['Next_Return'] * 100,
        mode='markers',
        marker=dict(
            color=merged['Next_Return'],
            colorscale='RdYlGn',
            size=8, showscale=True
        ),
        text=merged.index.strftime('%Y-%m-%d'),
        hovertemplate='Date: %{text}<br>Sentiment: %{x:.3f}<br>Next Return: %{y:.2f}%'
    ))
    fig.update_layout(template='plotly_dark', height=450,
                       title=f'🔗 News Sentiment vs Next-Day Return — {TARGET_TICKER}  (r={corr:.3f})',
                       xaxis_title='VADER Sentiment Score',
                       yaxis_title='Next-Day Return (%)')
    fig.show()
else:
    print('Not enough overlapping data for correlation. Try a more active ticker or wider date range.')

Not enough overlapping data for correlation. Try a more active ticker or wider date range.


## 5. Save Sentiment Data

In [6]:
news_df.to_csv('../data/news_sentiment.csv', index=False)
daily_sentiment.to_csv('../data/daily_sentiment.csv')
print('💾 Sentiment data saved.')
print('\n✅ All 4 notebooks complete! Launch the dashboard with:')
print('   cd dashboard && streamlit run app.py')

💾 Sentiment data saved.

✅ All 4 notebooks complete! Launch the dashboard with:
   cd dashboard && streamlit run app.py
